In [0]:
# ==========================================
# CẤU HÌNH VÀ BẢO MẬT (CHUẨN ENTERPRISE)
# ==========================================

# 1. Các thông số cấu hình chung (Không cần bảo mật)
EH_NAMESPACE = "evhns-clickstream-dev" 
EH_TOPIC = "clickstream-topic"
BOOTSTRAP_SERVERS = f"{EH_NAMESPACE}.servicebus.windows.net:9093"

STORAGE_ACCOUNT_NAME = "stclickstreamdev"
CONTAINER_NAME = "datalake"

# 2. Gọi API để móc chìa khóa từ Két sắt (Key Vault)
# dbutils.secrets.get sẽ trả về chuỗi đã bị mã hóa
EH_CONNECTION_STRING = dbutils.secrets.get(scope="clickstream_secrets", key="eventhub-conn-string")
STORAGE_ACCOUNT_KEY = dbutils.secrets.get(scope="clickstream_secrets", key="adls-access-key")

# 3. Ráp chìa khóa vào các cấu hình bảo mật
# Cấu hình Kafka SASL
EH_SASL = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EH_CONNECTION_STRING}";'

# Cấu hình quyền truy cập Azure Data Lake cho Spark
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    STORAGE_ACCOUNT_KEY
)
# Đường dẫn Data Lake
SILVER_TABLE_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/silver/events"

DIM_USER_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold/dim_user"
DIM_PRODUCT_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold/dim_product"
FACT_CLICKSTREAM_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold/fact_clickstream"

CHECKPOINT_GOLD_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/checkpoints/gold_events"
print("✅ Đã load cấu hình tầng Gold!")

In [0]:
# ==========================================
# CELL 2: KHỞI TẠO BẢNG (INITIALIZATION)
# ==========================================
from delta.tables import DeltaTable

print("Đang xây dựng cấu trúc bảng Star Schema...")

# 1. Bảng FACT
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS delta.`{FACT_CLICKSTREAM_PATH}` (
        event_id STRING, user_id STRING, product_id STRING, 
        session_id STRING, action STRING, event_timestamp TIMESTAMP
    ) USING DELTA
""")

# 2. Bảng DIM USER (SCD Type 1 - Ghi đè thông tin mới nhất)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS delta.`{DIM_USER_PATH}` (
        user_id STRING, device_type STRING, os STRING, 
        utm_source STRING, ip_address STRING
    ) USING DELTA
""")

# 3. Bảng DIM PRODUCT (SCD Type 2 - Lưu lại lịch sử thay đổi giá)
# Thêm 3 cột: is_current, valid_from, valid_to
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS delta.`{DIM_PRODUCT_PATH}` (
        product_id STRING, category STRING, price DOUBLE,
        is_current BOOLEAN, valid_from TIMESTAMP, valid_to TIMESTAMP
    ) USING DELTA
""")

print("Đã khởi tạo xong Fact và Dim!")

In [0]:
# ==========================================
# CELL 3: LOGIC XỬ LÝ SCD & STAR SCHEMA (FIXED)
# ==========================================

def process_gold_batch(microBatchDF, batchId):
    batch_spark = microBatchDF.sparkSession
    batch_df = microBatchDF.dropDuplicates(["event_id"])
    batch_df.createOrReplaceTempView("batch_vw")
    
    # ----------------------------------------------------
    # 1. XỬ LÝ FACT TABLE
    # ----------------------------------------------------
    batch_spark.sql(f"""
        INSERT INTO delta.`{FACT_CLICKSTREAM_PATH}`
        SELECT event_id, user_id, product_id, session_id, action, event_timestamp
        FROM batch_vw
    """)
    
    # ----------------------------------------------------
    # 2. XỬ LÝ DIM USER (SCD Type 1)
    # ----------------------------------------------------
    batch_spark.sql(f"""
        MERGE INTO delta.`{DIM_USER_PATH}` target
        USING (
            SELECT user_id, ANY_VALUE(device_type) as device_type, ANY_VALUE(os) as os, 
                   ANY_VALUE(utm_source) as utm_source, ANY_VALUE(ip_address) as ip_address 
            FROM batch_vw GROUP BY user_id
        ) source
        ON target.user_id = source.user_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    # ----------------------------------------------------
    # 3. XỬ LÝ DIM PRODUCT (SCD TYPE 2)
    # ----------------------------------------------------
    batch_spark.sql("""
        CREATE OR REPLACE TEMP VIEW latest_products_vw AS
        SELECT product_id, ANY_VALUE(category) as category, LAST(price) as price 
        FROM batch_vw GROUP BY product_id
    """)
    
    batch_spark.sql(f"""
        MERGE INTO delta.`{DIM_PRODUCT_PATH}` target
        USING latest_products_vw source
        ON target.product_id = source.product_id AND target.is_current = true
        WHEN MATCHED AND target.price != source.price THEN
            UPDATE SET is_current = false, valid_to = current_timestamp()
    """)
    
    batch_spark.sql(f"""
        MERGE INTO delta.`{DIM_PRODUCT_PATH}` target
        USING latest_products_vw source
        ON target.product_id = source.product_id AND target.is_current = true
        WHEN NOT MATCHED THEN
            INSERT (product_id, category, price, is_current, valid_from, valid_to)
            VALUES (source.product_id, source.category, source.price, true, current_timestamp(), cast('9999-12-31' as timestamp))
    """)

print("✅ Đã nạp hàm foreachBatch vào bộ nhớ!")

In [0]:
# ==========================================
# CELL 4: KÍCH HOẠT ĐỌC VÀ GHI
# ==========================================
print("Đang khởi động luồng Silver -> Gold...")
silver_stream_df = spark.readStream.format("delta").load(SILVER_TABLE_PATH)
# Dùng foreachBatch để đẩy dữ liệu qua cái hàm nhào nặn ở Cell 3
query_gold = (silver_stream_df.writeStream
    .foreachBatch(process_gold_batch)
    .option("checkpointLocation", CHECKPOINT_GOLD_PATH)
    .start()
)